In [0]:
# ─── Personal practice schema — never write directly to shared gbmart tables ────
# HOW TO GET A SCHEMA NAME: use a catalog/schema you already have CREATE permission on (ask your instructor if unsure), and pick something unique to you, e.g. main.virinchy_zorder_lab.
PRACTICE_SCHEMA = "harsh_kumar01_npmentorskool_onmicrosoft_com.practice_schema"   # ← replace with a schema you own
catalog = "harsh_kumar01_npmentorskool_onmicrosoft_com"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PRACTICE_SCHEMA}")

FACT_TABLE    = f"{PRACTICE_SCHEMA}.fact_sales_practice"
PRODUCT_TABLE = f"{PRACTICE_SCHEMA}.dim_product_practice"
DATE_TABLE    = f"{PRACTICE_SCHEMA}.dim_date_practice"
LIQUID_TABLE  = f"{PRACTICE_SCHEMA}.fact_sales_liquid_practice"

# SHALLOW CLONE: identical data to the real Gold tables right now, but a
# transaction history that's entirely yours — safe to rewrite repeatedly.
spark.sql(f"CREATE OR REPLACE TABLE {FACT_TABLE}    SHALLOW CLONE {catalog}.gold.fact_sales")
spark.sql(f"CREATE OR REPLACE TABLE {PRODUCT_TABLE} SHALLOW CLONE {catalog}.gold.dim_product")
spark.sql(f"CREATE OR REPLACE TABLE {DATE_TABLE}    SHALLOW CLONE {catalog}.gold.dim_date")

for t in [FACT_TABLE, PRODUCT_TABLE, DATE_TABLE]:
    print(f"{t}: {spark.table(t).count():,} rows")

harsh_kumar01_npmentorskool_onmicrosoft_com.practice_schema.fact_sales_practice: 377,866 rows
harsh_kumar01_npmentorskool_onmicrosoft_com.practice_schema.dim_product_practice: 500 rows
harsh_kumar01_npmentorskool_onmicrosoft_com.practice_schema.dim_date_practice: 11,344 rows


In [0]:
# repartition(200) forces the write into 200 separate output files,
# simulating the small-file state an incrementally-refreshed fact_sales
# reaches after many small daily MERGE runs — without waiting months to
# actually get there.
(
    spark.table(FACT_TABLE)
    .repartition(200)
    .write.format("delta").mode("overwrite")
    .saveAsTable(FACT_TABLE)
)
print("Practice fact table rewritten across 200 files.")

Practice fact table rewritten across 200 files.


In [0]:
detail = spark.sql(f"DESCRIBE DETAIL {FACT_TABLE}").collect()[0]
num_files = detail["numFiles"]
size_mb = detail["sizeInBytes"] / (1024 * 1024)
avg_file_mb = size_mb / num_files

print(f"Number of files : {num_files}")
print(f"Total size (MB) : {size_mb:.1f}")
print(f"Average file size (MB) : {avg_file_mb:.2f}")

Number of files : 200
Total size (MB) : 21.9
Average file size (MB) : 0.11


In [0]:
import time

def category_month_query(fact_table_name):
    """Shape 1: category/month revenue rollup — joins dim_product and dim_date."""
    return spark.sql(f"""
        SELECT p.category, d.month, SUM(f.Sales_amount) AS revenue
        FROM {fact_table_name} f
        JOIN {PRODUCT_TABLE} p ON f.Product_ID = p.product_id AND p.is_current = true
        JOIN {DATE_TABLE} d    ON f.Time_ID = d.date_key
        GROUP BY p.category, d.month
    """)

def regional_query(fact_table_name):
    """Shape 2: state/city regional rollup — joins dim_address (via a re-clone below)."""
    return spark.sql(f"""
        SELECT a.state, a.city, SUM(f.Sales_amount) AS revenue, COUNT(DISTINCT f.Order_ID) AS orders
        FROM {fact_table_name} f
        JOIN {PRACTICE_SCHEMA}.dim_address_practice a ON f.Address_ID = a.address_id
        GROUP BY a.state, a.city
    """)

def product_point_lookup(fact_table_name, sample_product_id, start_date_key, end_date_key):
    """Shape 3: narrow point lookup — one Product_ID, a date range on Time_ID."""
    return spark.sql(f"""
        SELECT f.Order_ID, f.Time_ID, f.Quantity_purchased, f.Sales_amount
        FROM {fact_table_name} f
        WHERE f.Product_ID = '{sample_product_id}'
          AND f.Time_ID BETWEEN {start_date_key} AND {end_date_key}
    """)

# dim_address_practice wasn't cloned in Scenario 1 -- Shape 2 needs it, so clone it now,
# still only ever from gbmart, still a one-time read-only operation.
spark.sql(f"CREATE OR REPLACE TABLE {PRACTICE_SCHEMA}.dim_address_practice SHALLOW CLONE harsh_kumar01_npmentorskool_onmicrosoft_com.gold.dim_address")

# Pick one real Product_ID and a real 30-day date_key range to use for Shape 3,
# so every learner's point lookup is grounded in data that actually exists.
sample_row = spark.table(FACT_TABLE).select("Product_ID", "Time_ID").limit(1).collect()[0]
SAMPLE_PRODUCT_ID = sample_row["Product_ID"]
SAMPLE_START_DATE_KEY = sample_row["Time_ID"]
SAMPLE_END_DATE_KEY = sample_row["Time_ID"] + 30

print(f"Using Product_ID={SAMPLE_PRODUCT_ID}, Time_ID range {SAMPLE_START_DATE_KEY}-{SAMPLE_END_DATE_KEY} for the point-lookup query.")

Using Product_ID=PRD-00001, Time_ID range 20230531-20230561 for the point-lookup query.


In [0]:
start = time.time()
result_count = category_month_query(FACT_TABLE).count()
baseline_category_seconds = time.time() - start

print(f"Result rows : {result_count}")
print(f"Baseline category/month query time : {baseline_category_seconds:.3f}s")

Result rows : 120
Baseline category/month query time : 4.090s


In [0]:
start = time.time()
result_count = regional_query(FACT_TABLE).count()
baseline_regional_seconds = time.time() - start

print(f"Result rows : {result_count}")
print(f"Baseline regional query time : {baseline_regional_seconds:.3f}s")

Result rows : 137
Baseline regional query time : 3.238s


In [0]:
start = time.time()
result_count = product_point_lookup(FACT_TABLE, SAMPLE_PRODUCT_ID, SAMPLE_START_DATE_KEY, SAMPLE_END_DATE_KEY).count()
baseline_point_seconds = time.time() - start

print(f"Result rows : {result_count}")
print(f"Baseline point-lookup query time : {baseline_point_seconds:.3f}s")

Result rows : 29
Baseline point-lookup query time : 1.883s


In [0]:
files_before = spark.sql(f"DESCRIBE DETAIL {FACT_TABLE}").collect()[0]["numFiles"]

spark.sql(f"OPTIMIZE {FACT_TABLE} ZORDER BY (Product_ID, Time_ID)")

files_after = spark.sql(f"DESCRIBE DETAIL {FACT_TABLE}").collect()[0]["numFiles"]
pct_reduction = round(100 * (files_before - files_after) / files_before, 1)

print(f"Files before : {files_before}")
print(f"Files after  : {files_after}")
print(f"Reduction    : {pct_reduction}%")

Files before : 200
Files after  : 1
Reduction    : 99.5%


In [0]:
start = time.time(); category_month_query(FACT_TABLE).count(); zorder_category_seconds = time.time() - start
start = time.time(); regional_query(FACT_TABLE).count(); zorder_regional_seconds = time.time() - start
start = time.time(); product_point_lookup(FACT_TABLE, SAMPLE_PRODUCT_ID, SAMPLE_START_DATE_KEY, SAMPLE_END_DATE_KEY).count(); zorder_point_seconds = time.time() - start

def pct_improvement(before, after):
    return round(100 * (before - after) / before, 1) if before > 0 else 0.0

print(f"{'Query':<20} {'Baseline':>10} {'Z-ORDER':>10} {'Improvement':>12}")
print(f"{'Category/Month':<20} {baseline_category_seconds:>9.3f}s {zorder_category_seconds:>9.3f}s {pct_improvement(baseline_category_seconds, zorder_category_seconds):>11}%")
print(f"{'Regional':<20} {baseline_regional_seconds:>9.3f}s {zorder_regional_seconds:>9.3f}s {pct_improvement(baseline_regional_seconds, zorder_regional_seconds):>11}%")
print(f"{'Point Lookup':<20} {baseline_point_seconds:>9.3f}s {zorder_point_seconds:>9.3f}s {pct_improvement(baseline_point_seconds, zorder_point_seconds):>11}%")

Query                  Baseline    Z-ORDER  Improvement
Category/Month           4.090s     2.045s        50.0%
Regional                 3.238s     1.082s        66.6%
Point Lookup             1.883s     0.682s        63.8%


In [0]:
spark.sql(f"DROP TABLE IF EXISTS {LIQUID_TABLE}")
spark.sql(f"""
    CREATE TABLE {LIQUID_TABLE}
    CLUSTER BY (Product_ID, Time_ID)
    AS SELECT * FROM {FACT_TABLE}
""")

# OPTIMIZE on a clustered table applies clustering incrementally -- no
# ZORDER BY clause needed; the table already knows its own cluster columns.
spark.sql(f"OPTIMIZE {LIQUID_TABLE}")
print(f"{LIQUID_TABLE} built and clustered on (Product_ID, Time_ID).")

harsh_kumar01_npmentorskool_onmicrosoft_com.practice_schema.fact_sales_liquid_practice built and clustered on (Product_ID, Time_ID).


In [0]:
start = time.time(); category_month_query(LIQUID_TABLE).count(); liquid_category_seconds = time.time() - start
start = time.time(); regional_query(LIQUID_TABLE).count(); liquid_regional_seconds = time.time() - start
start = time.time(); product_point_lookup(LIQUID_TABLE, SAMPLE_PRODUCT_ID, SAMPLE_START_DATE_KEY, SAMPLE_END_DATE_KEY).count(); liquid_point_seconds = time.time() - start

print(f"{'Query':<20} {'Z-ORDER':>10} {'Liquid':>10}")
print(f"{'Category/Month':<20} {zorder_category_seconds:>9.3f}s {liquid_category_seconds:>9.3f}s")
print(f"{'Regional':<20} {zorder_regional_seconds:>9.3f}s {liquid_regional_seconds:>9.3f}s")
print(f"{'Point Lookup':<20} {zorder_point_seconds:>9.3f}s {liquid_point_seconds:>9.3f}s")

Query                   Z-ORDER     Liquid
Category/Month           2.045s     1.633s
Regional                 1.082s     0.992s
Point Lookup             0.682s     0.502s


In [0]:
# A small, realistic append -- roughly what one day of new orders might look
# like landing via Day 9-10's incremental MERGE, applied identically to both
# the Z-ordered table and the Liquid-clustered table so the comparison is fair.
new_rows_df = spark.table(FACT_TABLE).limit(500)

new_rows_df.write.format("delta").mode("append").saveAsTable(FACT_TABLE)
new_rows_df.write.format("delta").mode("append").saveAsTable(LIQUID_TABLE)

print(f"{FACT_TABLE}: {spark.table(FACT_TABLE).count():,} rows after append")
print(f"{LIQUID_TABLE}: {spark.table(LIQUID_TABLE).count():,} rows after append")

harsh_kumar01_npmentorskool_onmicrosoft_com.practice_schema.fact_sales_practice: 378,366 rows after append
harsh_kumar01_npmentorskool_onmicrosoft_com.practice_schema.fact_sales_liquid_practice: 378,366 rows after append


In [0]:
start = time.time()
spark.sql(f"OPTIMIZE {FACT_TABLE} ZORDER BY (Product_ID, Time_ID)")
zorder_reoptimize_seconds = time.time() - start

start = time.time()
spark.sql(f"OPTIMIZE {LIQUID_TABLE}")
liquid_reoptimize_seconds = time.time() - start

print(f"Z-ORDER re-OPTIMIZE time  : {zorder_reoptimize_seconds:.3f}s")
print(f"Liquid re-OPTIMIZE time   : {liquid_reoptimize_seconds:.3f}s")

Z-ORDER re-OPTIMIZE time  : 7.702s
Liquid re-OPTIMIZE time   : 3.223s


In [0]:
from pyspark.sql import Row

summary_rows = [
    Row(query="Category/Month", baseline=round(baseline_category_seconds, 3), zorder=round(zorder_category_seconds, 3), liquid=round(liquid_category_seconds, 3)),
    Row(query="Regional",       baseline=round(baseline_regional_seconds, 3), zorder=round(zorder_regional_seconds, 3), liquid=round(liquid_regional_seconds, 3)),
    Row(query="Point Lookup",   baseline=round(baseline_point_seconds, 3),    zorder=round(zorder_point_seconds, 3),    liquid=round(liquid_point_seconds, 3)),
]

summary_df = spark.createDataFrame(summary_rows)
summary_df.show(truncate=False)

+--------------+--------+------+------+
|query         |baseline|zorder|liquid|
+--------------+--------+------+------+
|Category/Month|4.09    |2.045 |1.633 |
|Regional      |3.238   |1.082 |0.992 |
|Point Lookup  |1.883   |0.682 |0.502 |
+--------------+--------+------+------+

